# `Qwen3.6-35B-A6B-LoopHSSD` (v3.1) · Notebook 2: End-to-End Training & Whole-Model Pack Release

**Self-Resuming Multi-Session Pipeline (`Gate 0b → Gate 4`) + Low-Disk Streaming Whole-Model HF Release (Colab G4 · `RTX PRO 6000 Blackwell` `96 GiB` · `230 GB` SSD Safe)**

| Specialized Notebook | Purpose |
| :--- | :--- |
| [`colab_probe.ipynb`](colab_probe.ipynb) | **Zero-Training Diagnostic & Calibration Probes**: `Gate -1` + `Gate 0a` + `Gate 0p-1..4` (`7/7` passed, synced to `TassP0lar/Qwen3.6-35B-A3B-Loop64`) |
| **[`colab_train_e2e.ipynb`](colab_train_e2e.ipynb)** *(you are here)* | **End-to-End `Loop-HSSD v3.1` Training & Whole-Model Pack Release**: Resumable multi-stage evolution -> complete self-contained HF **Whole Model Pack** |
| [`colab_infer_eval.ipynb`](colab_infer_eval.ipynb) | **Shipped Whole-Model On-Weight Inference & Frontier `R=1/2/3` Eval**: `OptimizedLoopInferenceEngine`, `1M FP8` / `10M INT4` offload, frontier depth-curve suite |
| [`colab_train.ipynb`](colab_train.ipynb) | Baseline `64L` Track-A (`GDN-2` + `mHC`) single-notebook pipeline |

### Locked `s0_probe` Calibration & Staged Kill-Gate Evolution (`TassP0lar/Qwen3.6-35B-A3B-Loop64` → Shipped Whole Model Pack)
- **`Gate 0p` (`s0_probe.yaml`, Auto-Skipped when `completed` in `manifest.json`)**: Empirical Blackwell winners locked across all v3.1 stage configs — **`[0, 12) × 1 + [12, 36) × 3 + [36, 40) × 1 = 88L`** (`+0.4092` nats zero-shot loop delta vs `+1.0032` nats on `[8, 32)`), **`w_anchor = 0.85`** (`bridge_init: [0.05, 0.85, 0.10]`, `0.9739` manifold cosine, `0.7476` Pass-2 Jaccard), and pure **`1M YaRN 4.0×`** (`NLL = 2.6459`, `+13.7209` passkey margin, `100%` needle accuracy over `rerope`).
1. **`Gate 0b` (`s1_loop_a6b.yaml`)**: `R=2 (64L, ~4.46B) → R=3 (88L A6B, ~5.97B)` Subspace-Aligned `[12, 36)` Sandwich Loop Heal (`ReEntryBridge` `r=256`, `w_anchor=0.85`, zero-init `PassExitGate`, task-driven `PassRouterAdapters`, per-pass Q/O rank-64 LoRA, Track-A `GDN-2`, frozen `32.338B` MoE SHA-256 verified).
2. **`Gate 1` (`s1b_hssd_distill.yaml`)**: Track-B **`HSSD v3.1`** Horizon-Split Surprise-Gated Delta Memory KL Distillation (`[513, 576]` disjoint SWA + `PeakBoltzmannGate` + low-rank 2D phase `theta_low_rank=True` + routed root-surprise `s = g^2` + `M=8` Top-2 pages + `bfloat16` `81 MiB/layer` state + `segment_chunks=4` `256`-token checkpointed WY segments).
3. **`Gate 2a` (`s3_long_1m_lsa.yaml`)**: **`1M` GPU-Resident `FP8` SharedKV** + Per-Pass `D-LSA` (`num_query_passes=3`) + **Pure `YaRN 4.0×`** (`model.yarn_target_len=1048576`, `rerope.enabled=false`) + Long-Prefix (`512K–992K` `no_grad`) / Short-Suffix (`32K` `grad`) read-side calibration.
4. **`Gate 3` (`s2_joint_bptt.yaml`)**: Joint `32K` High-Density CoT SFT + **`4×32K = 128K` Differentiable K/V-Replay Truncated BPTT** (`SegmentedKVBPTTRunner`, Top-`2,048` prior-segment K/V re-projection).
5. **`Gate 4` (`s4_rl.yaml`)**: Verifiable-Reward **GRPO / RSI** on full `88L` Track-B `Loop-HSSD v3.1` + `1M YaRN` + sparse `D-LSA` (`DAPO-Math-17k` + `KernelBench L1–L2`, `rl.gate.depth=3`) with automatic held-out promotion/rollback gating.
6. **Final Release (`push_whole_model_pack(..., stream_low_disk=True)`)**: Folds optional `HotExpertDeltas`, serializes and streams sharded `model-00001-of-NNNNN.safetensors` **one `~4.5 GB` shard at a time** (`< 6 GB` peak temporary SSD!), and publishes `model.safetensors.index.json`, `config.json` (`auto_map` + `qloop_v3_arch`), standalone `configuration_qwen36_loophssd.py` & `modeling_qwen36_loophssd.py`, tokenizer, and Model Card to Hugging Face.

In [ ]:
REPO = "https://github.com/BaiTian6641/Qwen3.6-35B-A3B-Loop60.git"
BRANCH = "main"
%cd /content
!test -d /content/qloop/.git && test ! -d /content/qloop_repo && mv /content/qloop /content/qloop_repo || true
!test -d /content/qloop_repo || git clone -b $BRANCH $REPO /content/qloop_repo
%cd /content/qloop_repo
!git pull --ff-only
!bash scripts/colab_setup.sh

In [ ]:
import importlib
import os
import sys

if os.path.abspath("src") not in sys.path:
    sys.path.insert(0, os.path.abspath("src"))
if "qloop" in sys.modules and getattr(sys.modules["qloop"], "__file__", None) is None:
    for _k in [k for k in sys.modules if k == "qloop" or k.startswith("qloop.")]:
        del sys.modules[_k]
importlib.invalidate_caches()

from qloop.colab import (
    hub_status,
    q,
    reclaim_disk,
    resolve_base_model_name,
    run_v3_e2e_and_ship_pack,
    setup_env,
    tail_metrics,
)

BASE_PRECISION = "nvfp4"  # "nvfp4" (unsloth/Qwen3.6-35B-A3B-NVFP4, ~24 GiB) | "nvfp4-fast" (unsloth/Qwen3.6-35B-A3B-NVFP4-Fast, ~21 GiB) | "fp8" | "bf16"
HF_SYNC_REPO = "TassP0lar/Qwen3.6-35B-A3B-Loop64"        # stage-by-stage commit-history sync repo
HF_PACK_REPO = "TassP0lar/Qwen3.6-35B-A6B-LoopHSSD-Pack" # final self-contained Whole Model Pack repo
DEBUG_LEVEL = 1                                          # 1=normal | 2=NaN/Inf module hooks | 3=CUDA_LAUNCH_BLOCKING
TIME_BUDGET_HOURS = 21.5                                 # clean save + HF commit before 24h Colab timeout
CALIBRATED_OVERRIDES = ()                                # [12,36) 88L + HSSD v3.1 + 1M YaRN winners are locked in stage YAMLs

setup_env(
    hf_repo=HF_SYNC_REPO,
    run_root="/content/runs",
    hf_home="/content/hf_cache",
    base_precision=BASE_PRECISION,
)
q(f"doctor --freeze requirements/colab.lock.txt --model {resolve_base_model_name(BASE_PRECISION)} --debug {DEBUG_LEVEL}")
hub_status(HF_SYNC_REPO)

### 1 · Self-Resuming `Loop-HSSD v3.1` Pipeline + Streaming Whole-Model Pack Release
Pressing **`Runtime → Run all`** in any fresh Colab session queries `manifest.json` on `HF_SYNC_REPO`, automatically skips completed stages (including `s0_probe` once `completed` on Hub), pulls only the active `~1–3 GB` trainable delta, trains within `TIME_BUDGET_HOURS`, prunes superseded local files (`>= 106 GiB` SSD free), and upon final stage completion streams the complete sharded **Whole Model Pack** to `HF_PACK_REPO`.

In [ ]:
V3_STAGES = (
    "s0_probe",          # auto-skipped when status == 'completed' in remote HF manifest.json
    "s1_loop_a6b",       # [0,12)x1 + [12,36)x3 + [36,40)x1 = 88L A6B Subspace Loop Heal
    "s1b_hssd_distill",  # Track-B HSSD v3.1 (bf16 state, segment_chunks=4) KL Distillation
    "s3_long_1m_lsa",    # 1M YaRN 4.0x (rerope=false) + FP8 SharedKV + Per-Pass D-LSA
    "s2_joint_bptt",     # 32K CoT SFT + 4x32K (128K) Differentiable K/V-Replay TBPTT
    "s4_rl",             # 88L Track-B Loop-HSSD v3.1 Verifiable-Reward GRPO / RSI
)

pipeline_result = run_v3_e2e_and_ship_pack(
    hf_repo=HF_SYNC_REPO,
    pack_repo=HF_PACK_REPO,
    base_precision=BASE_PRECISION,
    stages=V3_STAGES,
    debug=DEBUG_LEVEL,
    time_budget_hours=TIME_BUDGET_HOURS,
    ship_whole_model_pack=True,
    stream_low_disk=True,
    merge_hot_experts=True,
    max_shard_size_bytes=4_500_000_000,
    overrides=CALIBRATED_OVERRIDES,
)
print("Pipeline & Release Result:", pipeline_result)
hub_status(HF_SYNC_REPO)

### 2 · Optional: Single-Stage Forensic Debug & Live Telemetry (`RUN_MANUAL_STAGE = True`)
Run or debug an individual `Loop-HSSD v3.1` stage (`s1_loop_a6b`, `s1b_hssd_distill`, `s3_long_1m_lsa`, `s2_joint_bptt`, `s4_rl`) with `--debug 2` (module-level NaN/Inf pinpointing + C++ traces) or `--debug 3` (`CUDA_LAUNCH_BLOCKING=1` + autograd anomaly + PyTorch memory snapshot).

In [ ]:
RUN_MANUAL_STAGE = False
MANUAL_STAGE = "s1_loop_a6b"  # s1_loop_a6b | s1b_hssd_distill | s3_long_1m_lsa | s2_joint_bptt | s4_rl
MANUAL_DEBUG = 2              # 2=NaN/Inf module hooks | 3=CUDA_LAUNCH_BLOCKING + anomaly + mem snapshot

if RUN_MANUAL_STAGE:
    q(
        f"train -c configs/stages/{MANUAL_STAGE}.yaml --debug {MANUAL_DEBUG} "
        f"-s model.name={resolve_base_model_name(BASE_PRECISION)} "
        f"-s checkpoint.sync.hf_repo={HF_SYNC_REPO}"
    )
    tail_metrics(MANUAL_STAGE)
    reclaim_disk(keep=MANUAL_STAGE)

### 3 · Optional: Manual Whole-Model Pack Export / Low-Disk Re-Stream (`RUN_MANUAL_PACK_PUSH = True`)
Automatically selects the latest completed stage (`s4_rl → s2_joint_bptt → s3_long_1m_lsa → s1b_hssd_distill → s1_loop_a6b`) from `HF_SYNC_REPO` / local `/content/runs` when `PACK_SOURCE_STAGE = "auto"` and `PACK_SOURCE_CONFIG = "auto"`, then streams a standalone multi-shard `safetensors` + custom HF modeling repository while keeping temporary SSD usage `< 6 GB`.

In [ ]:
RUN_MANUAL_PACK_PUSH = False
PACK_SOURCE_STAGE = "auto"   # "auto" picks latest completed: s4_rl -> s2_joint_bptt -> s3_long_1m_lsa -> s1b_hssd_distill -> s1_loop_a6b
PACK_SOURCE_CONFIG = "auto"  # "auto" resolves configs/stages/<selected_stage>.yaml (or pass explicit YAML path)
EXPORT_LOCAL_DIR = ""        # optional local path for `export_whole_model_pack` when non-empty

PACK_STAGE_PRIORITY = (
    "s4_rl",
    "s2_joint_bptt",
    "s3_long_1m_lsa",
    "s1b_hssd_distill",
    "s1_loop_a6b",
)

if RUN_MANUAL_PACK_PUSH:
    from qloop.cli import _stack_with_ckpt
    from qloop.colab import _local_stage_done
    from qloop.config import load_config
    from qloop.debug import crash_guard
    from qloop.export_pack import export_whole_model_pack, push_whole_model_pack
    from qloop.hub import HubSyncManager, resolve_stage_config_path

    run_root = os.environ.get("QLOOP_RUN_ROOT", "/content/runs")
    selected_stage = PACK_SOURCE_STAGE
    if selected_stage == "auto":
        manifest = HubSyncManager(repo_id=HF_SYNC_REPO, private=False).fetch_manifest()
        selected_stage = next(
            (st for st in PACK_STAGE_PRIORITY if manifest.is_completed(st) or _local_stage_done(run_root, st)),
            "",
        )
        if not selected_stage:
            raise RuntimeError(
                f"No completed v3 stage found on hf://{HF_SYNC_REPO} or under {run_root} "
                f"(checked {PACK_STAGE_PRIORITY})."
            )
    selected_cfg_path = (
        resolve_stage_config_path(selected_stage)
        if PACK_SOURCE_CONFIG == "auto"
        else PACK_SOURCE_CONFIG
    )
    print(f"Exporting Whole Model Pack from stage={selected_stage!r} using config={selected_cfg_path!r}")

    reclaim_disk(keep=selected_stage)
    with crash_guard(os.path.join(run_root, "pack_export")):
        cfg = load_config(
            selected_cfg_path,
            [
                f"model.name={resolve_base_model_name(BASE_PRECISION)}",
                f"checkpoint.sync.hf_repo={HF_SYNC_REPO}",
            ],
        )
        stack = _stack_with_ckpt(cfg, os.path.join(run_root, selected_stage))
        if EXPORT_LOCAL_DIR:
            local_manifest = export_whole_model_pack(
                stack,
                EXPORT_LOCAL_DIR,
                tokenizer=stack.tokenizer,
                max_shard_size_bytes=4_500_000_000,
                merge_hot_experts=True,
            )
            print("Exported local Whole Model Pack:", local_manifest)
        push_manifest = push_whole_model_pack(
            stack,
            HF_PACK_REPO,
            token=os.environ.get("HF_TOKEN"),
            stream_low_disk=True,
            max_shard_size_bytes=4_500_000_000,
            merge_hot_experts=True,
        )
        print("Streamed Whole Model Pack to HF Hub:", push_manifest)